In [ ]:
"""Welch t-test for the original Golub ALL/AML training microarray data.

Requirements:
    pip install numpy pandas scipy statsmodels

Usage:
    Place this script alongside data_set_ALL_AML_train.csv and run:
        python ALL_AML_welch_ttest.py

Outputs:
    ALL_AML_ttest_results.csv: one row per probe, ranked by p-value.
    ALL_AML_top50_samples.csv: illustrative selected-gene matrix, samples as rows.

CAUTION: The feature matrix was selected using ALL 38 training labels.
Do not cross-validate models on it. For cross-validation, rerun feature
selection within EACH training fold; only use the full-data selection if
assessing a completely untouched external test set.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests


DATA_PATH = Path.cwd() / "data" / "data_set_ALL_AML_train.csv"
OUTPUT_DIR = Path.cwd()

ALPHA = 0.05
N_TOP = 50


def main():
    df = pd.read_csv(DATA_PATH)

    # Original Golub training set: patient IDs 1..27 are ALL, 28..38 are AML.
    # The CSV's expression columns are NOT in numeric order.
    all_cols = [str(i) for i in range(1, 28)]
    aml_cols = [str(i) for i in range(28, 39)]
    sample_cols = all_cols + aml_cols
    required = ["Gene Description", "Gene Accession Number", *sample_cols]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Expected columns not found: {missing}")

    expression = df[sample_cols].apply(pd.to_numeric, errors="raise")
    all_expr = expression[all_cols].to_numpy(dtype=float)
    aml_expr = expression[aml_cols].to_numpy(dtype=float)

    # Each row is a separate gene probe, each column is a patient.
    t_stats, p_values = ttest_ind(
        all_expr, aml_expr, axis=1, equal_var=False, alternative="two-sided"
    )

    # Benjamini-Hochberg multiple-testing correction.
    q_values = np.full(len(p_values), np.nan, dtype=float)
    valid = np.isfinite(p_values)
    if not valid.any():
        raise ValueError("No valid t-tests; check expression columns.")
    q_values[valid] = multipletests(p_values[valid], method="fdr_bh")[1]

    mean_all = all_expr.mean(axis=1)
    mean_aml = aml_expr.mean(axis=1)
    results = pd.DataFrame({
        "Gene Description": df["Gene Description"],
        "Gene Accession Number": df["Gene Accession Number"],
        "mean_ALL": mean_all,
        "mean_AML": mean_aml,
        "mean_diff_ALL_minus_AML": mean_all - mean_aml,
        "t_statistic": t_stats,
        "p_value": p_values,
        "fdr_bh_q_value": q_values,
        "significant_fdr_0.05": q_values < ALPHA,
        "original_gene_row": np.arange(len(df)) + 1,
    }).sort_values("p_value", na_position="last").reset_index(drop=True)

    results_path = OUTPUT_DIR / "ALL_AML_ttest_results.csv"
    results.to_csv(results_path, index=False)

    # Illustrative model-ready table: each patient is a row, each probe a feature.
    # Keep the original row index to make column names unique even if accession IDs repeat.
    top = results.head(N_TOP)
    row_indexes = top["original_gene_row"].to_numpy(dtype=int) - 1
    selected = expression.iloc[row_indexes].T
    selected.columns = [
        f"probe_{row:04d}_{acc}"
        for row, acc in zip(top["original_gene_row"], top["Gene Accession Number"])
    ]
    selected.index.name = "sample_id"
    selected["cancer"] = ["ALL"] * len(all_cols) + ["AML"] * len(aml_cols)
    selected_path = OUTPUT_DIR / "ALL_AML_top50_samples.csv"
    selected.to_csv(selected_path)

    print(f"Genes/probes tested: {len(results)}")
    print(f"Patients: {len(all_cols)} ALL, {len(aml_cols)} AML")
    print(f"Nominal p < {ALPHA}: {int(np.sum(p_values < ALPHA))}")
    print(f"FDR-adjusted q < {ALPHA}: {int(np.sum(q_values < ALPHA))}")
    print("\nTop 5 probes by p-value:\n")
    print(results[[
        "Gene Accession Number", "t_statistic", "p_value", "fdr_bh_q_value"
    ]].head(5).to_string(index=False))
    print(f"\nSaved: {results_path}")
    print(f"Saved: {selected_path}")


if __name__ == "__main__":
    main()


Genes/probes tested: 7129
Patients: 27 ALL, 11 AML
Nominal p < 0.05: 1694
FDR-adjusted q < 0.05: 535

Top 7 probes by p-value:

Gene Accession Number  t_statistic      p_value  fdr_bh_q_value
     U22376_cds2_s_at     7.904300 8.388715e-09        0.000060
            X59417_at     6.803106 9.821652e-08        0.000350
            M55150_at    -8.091951 2.472593e-07        0.000464
          M31211_s_at     6.293715 2.917295e-07        0.000464
            L13278_at     6.281410 3.251349e-07        0.000464
          D26156_s_at     6.097113 5.184004e-07        0.000616
            M92287_at     6.217365 6.826528e-07        0.000661

Saved: /Users/ncby/Documents/Qiskit/QML_Leuk/QKernel-Leukemia/ALL_AML_ttest_results.csv
Saved: /Users/ncby/Documents/Qiskit/QML_Leuk/QKernel-Leukemia/ALL_AML_top50_samples.csv
